 # Décorateurs

Un décorateur est une fonction qui manipule une fonction ou une classe. C'est un élément du langage Python très pratique.

 Nous allons voir dans ces travaux pratiques comment les utiliser.

## Décorateurs de fonctions : fonctions qui manipulent des fonctions

Écrivez une fonction `squared`, qui prend en entrée une fonction et qui renvoie cette même fonction dont le résultat est passé au carré :

- la fonction originale prend toujours deux arguments
- la fonction originale produit toujours un nombre
- la fonction squared prend en entrée et produit en sortie **une fonction**

In [ ]:
def add(a, b):
  return a + b


def sub(a, b):
  return a - b


def mul(a, b):
  return a * b


def squared(function):
  return function  # Votre code ici


print(squared(add)(2, 2))  # Doit valoir 16
print(squared(sub)(10, 5))  # Doit valoir 25
print(squared(mul)(3, 4))  # Doit valoir 144

### Solution

In [ ]:
def squared(function):
  def new_function(a, b):
    return function(a, b) ** 2
  return new_function


print(squared(add)(2, 2))
print(squared(sub)(10, 5))
print(squared(mul)(3, 4))

## Syntaxe de décorateur

Pour appliquer un décorateur à une fonction, on peut utiliser la syntaxe suivante :

In [ ]:
@squared
def div(a, b):
  return a / b


print(div(8, 4))

Sans cette syntaxe qui utilise `@`, comment arriver au même résultat ?

In [ ]:
# Votre code ici

### Solution

In [ ]:
def div(a, b):
    return a / b


div = squared(div)


print(div(8, 4))

## Mesure du temps d'exécution d'une fonction

Créez un décorateur `timeit` qui affiche le temps d'exécution d'une fonction. Vous pourrez vous aider de la fonction [`time.time`](https://docs.python.org/fr/3/library/time.html#time.time)

In [ ]:
# Votre code ici

### Solution

In [ ]:
import time


def timeit(f):
  def new_f(*args, **kwargs):
    start = time.time()
    result = f(*args, **kwargs)
    print(f"Time elapsed during '{f.__name__}' call: {time.time() - start}s")
    return result
  return new_f


@timeit
def add(a, b):
  return a + b


add(1, 2)

## Enregistreur de classe

Il est souvent intéressant d'enregistrer des classes ou fonctions dans une structure de données (un registre), pour par exemple proposer un système de plugins.

Créez un décorateur de classe qui enregistre chaque classe à laquelle il est appliqué dans un dictionnaire, sous la clef `classe.__name__`.

In [ ]:
# Votre code ici

En plus d'ajouter le décorateur aux classes que vous souhaitez enregistrer, quelle étape importante est nécessaire pour que l'enregistrement se fasse ?

*Votre réponse*

### Solution

Ce pattern, appelé *Registry* en anglais, permet à une librairie d'enregistrer du code client pour pouvoir ensuite l'exécuter à un moment approprié.

Cela se passe souvent en trois étapes :

1. Mise à disposition d'un décorateur pour enregistrer une fonction ou une classe dans le code client
2. Utilisation du décorateur côté client
3. Utilisation des éléments enregistrés côté librairie

Par exemple, pour l'étape 1. :

In [ ]:
from collections.abc import Callable
from typing import Protocol


class SupportsProcess(Protocol):
  def process(self) -> None: ...


_plugins = {}


def register[T: SupportsProcess](cls: type[T]) -> type[T]:
  _plugins[cls.__name__] = cls
  return cls

Puis pour l'étape 2. :

In [ ]:
@register
class CustomClientProcessor(SupportsProcess):
  def process(self) -> None:
    print("Print depuis la méthode process custom du client")

Attention, il est nécessaire de charger le module qui contient la classe (sinon le décorateur ne sera pas exécuté et la classe ne sera pas enregistrée).

Enfin, pour l'étape 3. :

In [ ]:
for name, cls in _plugins.items():
  print(f"Appel de la méthode process de {name}")
  cls().process()

## Définition d'un décorateur avec argument

Reprenez l'exercice précédent mais cette fois le décorateur doit récupérer un argument `name` à utiliser comme clef de registre à la place de `function.__name__`.

In [ ]:
# Votre code ici

### Solution

In [ ]:
from collections.abc import Callable
from typing import Any, TypeVar


_plugins = {}


T = TypeVar("T")


def register(name: str) -> Callable[[type[T]], type[T]]:
  def decorator(cls: type[T]) -> type[T]:
    _plugins[name] = cls
    return cls
  return decorator


@register("B")
class A:
  pass


_plugins

## Utilisation de [`functools.wraps`](https://docs.python.org/fr/3/library/functools.html#functools.wraps)

Essayez de manipuler les fonctions modifiées dans le reste du TP et de trouver leur nom (attribut `__name__`) ou encore leur doc (attribut `__doc__`). Que remarquez-vous ?

*Votre réponse*

Utilisez `functools.wraps` sur l'exemple donné du décorateur `timeit` avec argument pour régler ce problème.

In [ ]:
# Votre code ici

### Solution

In [ ]:
import functools
import time


def timeit(unit="s"):
  def decorator(function):
    @functools.wraps(function)
    def new_f(*args, **kwargs):
      start = time.time()
      result = function(*args, **kwargs)
      duration = time.time() - start
      if unit == "ms":
        duration *= 1000
      elif unit != "s":
        raise ValueError("Can only use s or ms as unit argument")
      print(f"Time elapsed during '{function.__name__}' call: "
            f"{duration}{unit}")
      return result
    return new_f
  return decorator


@timeit(unit="ms")
def add(a, b):
  """Add two integers."""
  return a + b


print(add.__name__, add.__doc__)